# 04. 차선 영역 분할 모델 학습 (Colab GPU, 실험 — handoff 23장)

차선 마모 판정(밝기 대비)이 횡단보도·노면 글자·화살표를 차선으로 잡는 헛것(handoff 17장, gn_b 확정 중 84~93%)을 줄이려고,
**측정할 영역을 진짜 차선으로 한정하는 YOLO11n-seg** 를 학습한다. 판정 방식은 그대로, 영역만 좁힌다.

| 항목 | 값 |
|---|---|
| 데이터 | AI Hub 087 「도로 로드마크 인식을 위한 주행 영상 데이터」 검증 묶음 VS4/VL4 (비온전·복합: 낮·밤·비·안개) 3,281장 |
| 만드는 곳 | Mac: `uv run python -m daboyeo_ai.data.lane_seg ... --zip data/lane_seg.zip` → Drive `daboyeo_data/parts/lane_seg.zip` |
| 클래스 | 0 `lane` (중앙선·차선·버스전용·길가장자리·진로변경제한·주정차금지선), 1 `marking` (화살표·글자·횡단보도·정지선·안전지대 등) |
| 나누기 | 주행 + 연속 50 프레임 묶음 단위로 train 85% / val 15%. test 없음 — 진짜 시험은 시연 클립 gn_a·gn_b (Mac) |
| 결과 | Drive `models/lane_seg_v1.onnx` · `.pt` · `.json` |

**미리 할 것**: **런타임 → 런타임 유형 변경 → T4 GPU**. 세션이 끊겨도 Drive `runs/lane_seg_v1/weights/last.pt` 에서 이어진다 (위에서부터 다시 실행).

In [ ]:
# ===== 설정 =====
DRIVE_ROOT = "/content/drive/MyDrive/daboyeo_data"
REPO_URL = "https://github.com/ParkGyeom/daboyeo.git"

VERSION = "lane_seg_v1"
PART = "lane_seg"         # Drive parts/lane_seg.zip
BASE_MODEL = "yolo11n-seg.pt"   # COCO 사전학습 n급 (블랙박스 CPU 속도)
EPOCHS = 80
PATIENCE = 20
IMGSZ = 640
BATCH = 32                # T4 OOM 이면 16
FRACTION = 1.0            # 빠른 시험은 0.1

RUNS = f"{DRIVE_ROOT}/runs"
MODELS = f"{DRIVE_ROOT}/models"
DATASET = "/content"      # zip 안이 lane_seg/... 라 /content/lane_seg 에 풀린다

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib, json, shutil, subprocess, sys, time, zipfile
for d in (RUNS, MODELS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU 로 바꿔야 한다"

### 코드 설치 (`ultralytics` + 레포)

In [ ]:
REPO = "/content/daboyeo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
!pip install -q ultralytics onnx onnxslim onnxruntime -e {REPO}/contracts -e {REPO}/ai
sys.path[:0] = [f"{REPO}/contracts", f"{REPO}/ai"]
from daboyeo_ai.data.lane_seg import SEG_CLASS_NAMES, write_data_yaml
from daboyeo_ai.inference.lane_seg import LaneSegmenter
print(SEG_CLASS_NAMES)

## 1. 데이터 풀기

In [ ]:
from collections import Counter
import pandas as pd

zp = pathlib.Path(DRIVE_ROOT) / "parts" / f"{PART}.zip"
assert zp.exists(), f"Drive 에 없음: {zp} — Mac 에서 만든 lane_seg.zip 을 parts/ 에 올려야 한다"
root = pathlib.Path(DATASET) / PART
marker = pathlib.Path(DATASET) / f".done_{PART}"
if not marker.exists():
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(DATASET)
    marker.touch()
data_yaml = write_data_yaml(root)
print(json.loads((root / "stats.json").read_text())["stats"])

summary = Counter()
for txt in (root / "labels").glob("*/*.txt"):
    split = txt.parent.name
    summary[(split, "images")] += 1
    for line in txt.read_text().splitlines():
        summary[(split, SEG_CLASS_NAMES[int(line.split()[0])])] += 1
counts = pd.Series(summary).unstack(0).fillna(0).astype(int)
display(counts)
assert counts.get("val", pd.Series(dtype=int)).get("images", 0) > 0, "val 이미지가 없다"

## 2. 학습

- 처음: COCO 사전학습 `yolo11n-seg.pt` 에서 시작. 다시 실행하면 Drive `last.pt` 에서 이어서.
- 증강: 좌우 반전 O, 상하 반전 X, 밝기 변화 크게 (젖은 노면·야간). 모자이크는 마지막 10 epoch 에 끈다 (기본값).

In [ ]:
from ultralytics import YOLO
import torch

run_dir = pathlib.Path(RUNS) / VERSION
last = run_dir / "weights" / "last.pt"

def is_finished(ckpt_path):
    # 끝난 학습을 resume 하면 Ultralytics 가 COCO 샘플로 새 학습을 시작한다 — 미리 막는다 (지우지 말 것)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ck.get("epoch", -1) == -1 or ck.get("optimizer") is None

t = time.time()
if last.exists() and is_finished(last):
    print(f"{VERSION} 학습은 이미 끝났다 — 건너뜀 (다시 하려면 VERSION 을 바꾸거나 {run_dir} 를 지운다)")
elif last.exists():
    print("중단된 학습을 이어서:", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=str(data_yaml), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
        project=RUNS, name=VERSION, exist_ok=True, seed=0, workers=2,
        fliplr=0.5, flipud=0.0, hsv_v=0.5, plots=True,
    )
print(f"{(time.time() - t) / 60:.1f} 분")

## 3. 평가 (val, 클래스별 마스크 P·R·mAP50)

In [ ]:
best_pt = run_dir / "weights" / "best.pt"
best = YOLO(str(best_pt))
m = best.val(data=str(data_yaml), split="val", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
rows = []
for i, c in enumerate(m.seg.ap_class_index):
    rows.append(dict(cls=best.names[int(c)], mask_P=m.seg.p[i], mask_R=m.seg.r[i], mask_mAP50=m.seg.ap50[i],
                     box_mAP50=m.box.ap50[i]))
rows.append(dict(cls="all", mask_P=m.seg.mp, mask_R=m.seg.mr, mask_mAP50=m.seg.map50, box_mAP50=m.box.map50))
metrics_df = pd.DataFrame(rows).round(3)
display(metrics_df)

## 4. ONNX 변환 + 결과 일치 확인

블랙박스(Mac)에서는 `LaneSegmenter`(우리 전처리·후처리)로 돌린다. 같은 ONNX 를 Ultralytics 로 돌린 **차선 마스크와 겹침(IoU)** 을
val 10장에서 잰다. 평균 IoU 0.9 이상이면 OK (경계 1~2 픽셀 차이는 정상).

In [ ]:
import cv2, numpy as np
onnx_path = pathlib.Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
ours = LaneSegmenter(onnx_path, threads=2, conf=0.25, iou=0.5)
ref_model = YOLO(str(onnx_path), task="segment")

val_imgs = sorted((root / "images" / "val").glob("*.jpg"))[::max(1, len(list((root / "images" / "val").glob("*.jpg"))) // 10)][:10]
ious = []
for p in val_imgs:
    img = cv2.imread(str(p))
    r = ref_model.predict(img, imgsz=IMGSZ, conf=0.25, iou=0.5, retina_masks=True, verbose=False)[0]
    ref = np.zeros(img.shape[:2], bool)
    if r.masks is not None:
        for c, mk in zip(r.boxes.cls.int().tolist(), r.masks.data.cpu().numpy()):
            if c == 0:
                ref |= mk > 0.5
    mine = ours.lane_mask(img) > 0
    union = (ref | mine).sum()
    ious.append(1.0 if union == 0 else float((ref & mine).sum() / union))
parity = dict(images=len(val_imgs), mean_iou=round(float(np.mean(ious)), 3), min_iou=round(float(np.min(ious)), 3))
print(parity, "OK" if parity["mean_iou"] >= 0.9 else "⚠️ 0.9 미만 — 이 출력을 공유할 것")

In [ ]:
# CPU 속도 참고값 (Colab CPU). 정식 측정은 Mac 에서
speed = {}
frames = [cv2.imread(str(p)) for p in val_imgs]
for threads in (1, 2):
    s = LaneSegmenter(onnx_path, threads=threads)
    s.lane_mask(frames[0])
    t = time.perf_counter()
    for f in frames * 3:
        s.lane_mask(f)
    speed[f"{threads}thread_ms"] = round((time.perf_counter() - t) / (len(frames) * 3) * 1000, 1)
print(speed)

## 5. Drive `models/` 에 저장

In [ ]:
import datetime
out_onnx = pathlib.Path(MODELS) / f"{VERSION}.onnx"
shutil.copy(onnx_path, out_onnx)
shutil.copy(best_pt, pathlib.Path(MODELS) / f"{VERSION}.pt")
results_csv = run_dir / "results.csv"
card = {
    "version": VERSION,
    "file": out_onnx.name,
    "task": "segment",
    "classes": list(SEG_CLASS_NAMES),
    "input_size": IMGSZ,
    "recommended_conf": 0.25,
    "base_model": BASE_MODEL,
    "data": "AI Hub 087 VS4/VL4 (lane_seg.zip)",
    "train_counts": counts.to_dict(),
    "epochs_trained": len(pd.read_csv(results_csv)) if results_csv.exists() else None,
    "metrics_val": metrics_df.set_index("cls").to_dict("index"),
    "onnx_parity": parity,
    "colab_cpu_speed": speed,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}
(pathlib.Path(MODELS) / f"{VERSION}.json").write_text(json.dumps(card, ensure_ascii=False, indent=1, default=float))
print(json.dumps({k: card[k] for k in ("file", "epochs_trained", "metrics_val", "onnx_parity", "colab_cpu_speed")},
                 ensure_ascii=False, indent=1, default=float))
!ls -la {MODELS}

## 다음

1. Drive `daboyeo_data/models/lane_seg_v1.onnx` 와 `.json` 을 Mac 의 `ai/models/` 에 내려받는다 (git 에 안 올림).
2. Mac 에서 시연 클립 gn_a·gn_b 재평가 (handoff 23장).